<a href="https://colab.research.google.com/github/Abhinav9895/Generative-AI-Internship/blob/main/Day7/MovieReview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import re
df=pd.read_csv("/content/IMDB Dataset.csv",encoding='latin1', engine='python', on_bad_lines='skip')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [4]:
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})

In [5]:
negation_words=["not good","not bad","not great","don't like","didn't like","never liked","wasn't good","isn't good","no good"]
def clean_text(text):
  text=text.lower()
  text=re.sub(r"[^a-zA-Z\s']","",text)

  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace("","_"))

  return text

In [6]:
df['review']=df['review'].apply(clean_text)

In [7]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    df['review'],df['sentiment'],test_size=0.2,random_state=42
)

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
vocab_size=20000
meax_len=250

tokenizer=Tokenizer(num_words=vocab_size,oov_token="<OOV")
tokenizer.fit_on_texts(x_train)
x_train_seq=tokenizer.texts_to_sequences(x_train)
x_test_seq=tokenizer.texts_to_sequences(x_test)

x_train_pad=pad_sequences(x_train_seq,maxlen=meax_len,padding='post')
x_test_pad=pad_sequences(x_test_seq,maxlen=meax_len,padding='post')

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout


model=Sequential([
    Embedding(vocab_size,128,input_length=meax_len),
    LSTM(128,dropout=0.3,recurrent_dropout=0.2),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
history=model.fit(x_train_pad,y_train,epochs=5,batch_size=62,validation_split=0.2)

Epoch 1/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 422s 816ms/step - accuracy: 0.8982 - loss: 0.2671 - val_accuracy: 0.8805 - val_loss: 0.3079
Epoch 2/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 434s 801ms/step - accuracy: 0.9394 - loss: 0.1727 - val_accuracy: 0.8805 - val_loss: 0.3229
Epoch 3/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 447s 810ms/step - accuracy: 0.9629 - loss: 0.1167 - val_accuracy: 0.8810 - val_loss: 0.3757
Epoch 4/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 440s 807ms/step - accuracy: 0.9766 - loss: 0.0807 - val_accuracy: 0.8789 - val_loss: 0.4171
Epoch 5/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 432s 788ms/step - accuracy: 0.9847 - loss: 0.0558 - val_accuracy: 0.8742 - val_loss: 0.4691


In [13]:
loss,acc=model.evaluate(x_test_pad,y_test)
print("Test Accuracy:",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 32s 100ms/step - accuracy: 0.8770 - loss: 0.4509
Test Accuracy: 0.8769999742507935


In [14]:
def predict_sentiment(review):
  review=clean_text(review)
  sequence=tokenizer.texts_to_sequences([review])
  padded_sequence=pad_sequences(sequence,maxlen=meax_len,padding='post')
  prediction=model.predict(padded_sequence)[0][0]
  print("\nReview",review)
  print("Score",prediction)
  sentiment="positive" if prediction>0.5 else "negative"
  return sentiment

In [17]:
predict_sentiment("This movie is bad ")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step

Review this movie is bad 
Score 0.11885336


'negative'